In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_F.xpt"
csv_file = DATA_DIR / "BMX_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve exactly what pandas decoded from XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 BMX_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMXWT",
    "BMIWT",
    "BMXRECUM",
    "BMIRECUM",
    "BMXHEAD",
    "BMIHEAD",
    "BMXHT",
    "BMIHT",
    "BMXBMI",
    "BMXLEG",
    "BMILEG",
    "BMXARML",
    "BMIARML",
    "BMXARMC",
    "BMIARMC",
    "BMXWAIST",
    "BMIWAIST",
    "BMXTRI",
    "BMITRI",
    "BMXSUB",
    "BMISUB"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BMX_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10253:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 23:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,253 rows x 23 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# The attached BMX_F does NOT contain this artifact.
#
# Do NOT replace anything automatically.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Tiny XPORT artifacts were unexpectedly "
        "found in BMX_F. Do not automatically replace."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifact exists."
)


# ============================================================
# 5. IDENTIFY GENUINE DECIMAL MEASUREMENTS
# ============================================================

#
# CRITICAL FIDELITY RULE
# ------------------------------------------------------------
#
# Body measurements are genuine continuous values.
#
# DO NOT:
#
#     round them
#     truncate them
#     convert them to Int64
#
# Values such as:
#
#     87.4
#     164.7
#     32.22
#     41.5
#
# must remain exactly as supplied.
#

print("\n--- GENUINE DECIMAL CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional values"
        )


# ============================================================
# 6. VALIDATE EXPECTED DECIMAL VARIABLES
# ============================================================

#
# These 11 variables are physical measurements
# and legitimately contain decimal values.
#

DECIMAL_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXTRI",
    "BMXSUB"
]


if fractional_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs "
        "from validated BMX_F pattern.\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{fractional_columns}"
    )


print(
    "\nPASS: Genuine decimal measurement variables:",
    len(DECIMAL_COLUMNS)
)


print(
    "PASS: Measurement decimals will NOT be rounded "
    "or converted to integers."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. FORMAT ONLY WHOLE-NUMBER VARIABLES
# ============================================================

#
# Only identifier/status/comment-code variables
# are converted to Int64.
#
# The 11 physical measurement variables above
# remain floats.
#
# NO .round() is used.
#

INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in DECIMAL_COLUMNS
]


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN/status/comment fields formatted "
    "without unnecessary .0."
)


print(
    "PASS: Body measurement values remain unchanged."
)


# ============================================================
# 9. CDC CHECK - BMDSTATS
# ============================================================

#
# CDC:
#
# 1 = Complete data for age group       8,608
# 2 = Partial: height/weight only         337
# 3 = Other partial exam                1,231
# 4 = No body measures exam data           77
#

EXPECTED_BMDSTATS = {
    1: 8608,
    2: 337,
    3: 1231,
    4: 77
}


actual = (
    df["BMDSTATS"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- BMDSTATS ---")


if (
    actual != EXPECTED_BMDSTATS
    or
    int(
        df["BMDSTATS"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "BMDSTATS differs from CDC.\n"
        f"Expected: {EXPECTED_BMDSTATS}\n"
        f"Actual: {actual}"
    )


print(
    "PASS: BMDSTATS matches CDC."
)


# ============================================================
# 10. CDC CHECK - MEASUREMENT RANGES
# ============================================================

#
# Format:
#
# variable:
# (
#     minimum,
#     maximum,
#     expected non-missing count,
#     expected missing count
# )
#
# These are the released CDC ranges.
#

CDC_MEASUREMENTS = {

    "BMXWT": (
        2.7,
        239.4,
        10162,
        91
    ),

    "BMXRECUM": (
        45.0,
        114.8,
        1206,
        9047
    ),

    "BMXHEAD": (
        32.5,
        47.3,
        238,
        10015
    ),

    "BMXHT": (
        79.1,
        202.7,
        9420,
        833
    ),

    "BMXBMI": (
        12.58,
        84.87,
        9412,
        841
    ),

    "BMXLEG": (
        20.4,
        51.2,
        7755,
        2498
    ),

    "BMXARML": (
        9.2,
        47.0,
        9708,
        545
    ),

    "BMXARMC": (
        10.2,
        61.0,
        9699,
        554
    ),

    "BMXWAIST": (
        40.7,
        179.0,
        8973,
        1280
    ),

    "BMXTRI": (
        2.6,
        41.1,
        9358,
        895
    ),

    "BMXSUB": (
        3.2,
        40.4,
        8776,
        1477
    )
}


print("\n--- CDC MEASUREMENT CHECKS ---")


for col, (
    expected_min,
    expected_max,
    expected_nonmissing,
    expected_missing
) in CDC_MEASUREMENTS.items():

    s = df[col]


    actual_min = s.min()
    actual_max = s.max()

    actual_nonmissing = int(
        s.notna()
        .sum()
    )

    actual_missing = int(
        s.isna()
        .sum()
    )


    if not np.isclose(
        actual_min,
        expected_min,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col} minimum differs from CDC.\n"
            f"Expected: {expected_min}\n"
            f"Actual: {actual_min}"
        )


    if not np.isclose(
        actual_max,
        expected_max,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col} maximum differs from CDC.\n"
            f"Expected: {expected_max}\n"
            f"Actual: {actual_max}"
        )


    if actual_nonmissing != expected_nonmissing:

        raise ValueError(
            f"{col} non-missing count differs from CDC.\n"
            f"Expected: {expected_nonmissing}\n"
            f"Actual: {actual_nonmissing}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual: {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 11. CDC CHECK - COMMENT CODES
# ============================================================

#
# Comment variables are categorical whole-number codes.
#
# They should NOT contain .0 in the final CSV.
#

CDC_COMMENT_CODES = {

    "BMIWT": {
        "values": {
            1: 21,
            3: 333,
            4: 26
        },
        "missing": 9873
    },

    "BMIRECUM": {
        "values": {
            1: 13,
            3: 2
        },
        "missing": 10238
    },

    "BMIHEAD": {
        "values": {},
        "missing": 10253
    },

    "BMIHT": {
        "values": {
            1: 45,
            3: 258
        },
        "missing": 9950
    },

    "BMILEG": {
        "values": {
            1: 359
        },
        "missing": 9894
    },

    "BMIARML": {
        "values": {
            1: 406
        },
        "missing": 9847
    },

    "BMIARMC": {
        "values": {
            1: 416
        },
        "missing": 9837
    },

    "BMIWAIST": {
        "values": {
            1: 479
        },
        "missing": 9774
    },

    "BMITRI": {
        "values": {
            1: 617,
            2: 143
        },
        "missing": 9493
    },

    "BMISUB": {
        "values": {
            1: 1230,
            2: 111
        },
        "missing": 8912
    }
}


print("\n--- CDC COMMENT CODE CHECKS ---")


for col, expected in CDC_COMMENT_CODES.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual: {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual: {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 12. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# IMPORTANT:
#
# BMX_F contains NO validated tiny-zero corrections.
#
# Therefore EVERY numeric value should remain exactly
# equal to the original XPT numeric value.
#
# Int64 conversion changes only CSV representation of
# identifiers/status/comment codes:
#
#     51624.0 -> 51624
#
# It does not change the numeric value.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    if col in DECIMAL_COLUMNS:

        current_values = (
            df[col]
            .to_numpy(
                dtype=float
            )
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 23 variables preserve "
    "the original XPT numeric values exactly."
)


# ============================================================
# 13. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format
# NO rounding
#
# Continuous body measurements remain floats.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 14. RE-READ EXPORTED CSV
# ============================================================

#
# round_trip precision allows exact verification
# of floating-point measurement values.
#

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 15. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 23 CSV columns match "
    "source XPT numeric values exactly."
)


# ============================================================
# 16. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 17. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny-value artifacts."
)


# ============================================================
# 18. WHOLE-NUMBER .0 FORMAT CHECK
# ============================================================

#
# IMPORTANT:
#
# DO NOT search the entire CSV for ".0".
#
# Measurements are continuous variables.
#
# A value such as:
#
#     BMXARML = 40.0
#     BMXWT   = 17.0
#
# is legitimate in a measurement column and should
# remain represented as a decimal measurement.
#
# We only prohibit .0 in INTEGER_COLUMNS.
#

print("\n--- WHOLE-NUMBER FORMAT CHECK ---")


dot_zero_errors = []


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(
        f
    )


    for row_number, row in enumerate(
        reader,
        start=2
    ):

        for col in INTEGER_COLUMNS:

            value = row[col]


            if (
                value != ""
                and
                value.endswith(".0")
            ):

                dot_zero_errors.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


                if len(
                    dot_zero_errors
                ) >= 20:

                    break


        if len(
            dot_zero_errors
        ) >= 20:

            break


print(
    "Unnecessary '.0' errors:",
    len(
        dot_zero_errors
    )
)


if dot_zero_errors:

    print(
        dot_zero_errors
    )

    raise ValueError(
        "Unexpected .0 formatting remains "
        "in whole-number columns."
    )


print(
    "PASS: SEQN/status/comment codes "
    "have no unnecessary .0."
)


# ============================================================
# 19. EXACT MEASUREMENT DECIMAL FIDELITY
# ============================================================

#
# Explicitly verify each continuous measurement
# variable separately.
#
# These must survive:
#
# XPT
# -> CSV
# -> pandas re-read
#
# with zero numeric change.
#

print("\n--- MEASUREMENT DECIMAL FIDELITY ---")


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Body measurement values changed in:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All",
    len(DECIMAL_COLUMNS),
    "measurement variables preserved exactly."
)


# ============================================================
# 20. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 21. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "BMX_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Tiny-value corrections: NONE"
)


print(
    "Continuous measurement variables preserved:",
    len(DECIMAL_COLUMNS)
)


print(
    "Whole-number identifier/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in whole-number fields: 0 expected"
)


print(
    "Measurement decimal values changed: 0 expected"
)

NHANES 2009-2010 BMX_F XPT -> CSV
Rows: 10,253
Columns: 23
PASS: Structure = 10,253 rows x 23 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifact exists.

--- GENUINE DECIMAL CHECK ---
BMXWT: 9,123 fractional values
BMXRECUM: 1,096 fractional values
BMXHEAD: 210 fractional values
BMXHT: 8,511 fractional values
BMXBMI: 9,339 fractional values
BMXLEG: 6,190 fractional values
BMXARML: 6,599 fractional values
BMXARMC: 8,048 fractional values
BMXWAIST: 7,397 fractional values
BMXTRI: 6,761 fractional values
BMXSUB: 6,417 fractional values

PASS: Genuine decimal measurement variables: 11
PASS: Measurement decimals will NOT be rounded or converted to integers.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: SEQN/status/comment fields formatted without unnecessary .0.
PASS: Body measurement values remain unchanged.

--- BMDSTATS ---
PASS: BMDSTATS matches CDC.

